# 🏗️ Day 2 — Model Architecture
**Vision System Capstone v3** | ResNet from Scratch + SE Blocks

This notebook covers:
1. Build ResNet from config
2. Forward pass sanity check (shape + dtype)
3. Parameter count + FLOPs
4. Stage output shapes
5. Architecture diagram
6. SE Block visualisation
7. Weight initialisation check
8. Grad-CAM hook verification

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import yaml
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
print('✓ Setup done')

In [ ]:
# ── Load configs ──────────────────────────────────────────────────────────────
with open('../configs/model_config.yaml') as f:
    MCFG = yaml.safe_load(f)
with open('../configs/data_config.yaml') as f:
    DCFG = yaml.safe_load(f)

print('Model config:')
print(f'  depth      : {MCFG["arch"]["depth"]}')
print(f'  channels   : {MCFG["arch"]["channels"]}')
print(f'  SE block   : {MCFG["blocks"]["se_block"]["enabled"]}')
print(f'  reduction  : {MCFG["blocks"]["se_block"]["reduction_ratio"]}')
print(f'  norm       : {MCFG["blocks"]["norm_layer"]}')
print(f'  activation : {MCFG["blocks"]["activation"]}')
print(f'  dropout    : {MCFG["head"]["dropout"]}')
print()
print(f'Data config:')
print(f'  dataset    : {DCFG["dataset"]["name"]}')
print(f'  num_classes: {DCFG["dataset"]["num_classes"]}')
print(f'  image_size : {DCFG["image"]["size"]}')

---
## 1. Build Model

In [ ]:
# ── Build from config ─────────────────────────────────────────────────────────
from models.resnet import build_model, ResNet

model = build_model(
    model_cfg_path='../configs/model_config.yaml',
    data_cfg_path ='../configs/data_config.yaml',
).to(device)

print(f'Model built: ResNet-{MCFG["arch"]["depth"]} | SE={MCFG["blocks"]["se_block"]["enabled"]}')
print(f'Total trainable params: {model.count_params():,}')

---
## 2. Forward Pass Sanity Check

In [ ]:
# ── Forward pass ──────────────────────────────────────────────────────────────
IMG_SIZE    = DCFG['image']['size']
NUM_CLASSES = DCFG['dataset']['num_classes']
BATCH_SIZE  = 4

dummy_input = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE).to(device)

model.eval()
with torch.no_grad():
    logits = model(dummy_input)

# Assertions
assert logits.shape == (BATCH_SIZE, NUM_CLASSES), \
    f'Expected ({BATCH_SIZE}, {NUM_CLASSES}), got {logits.shape}'
assert not torch.isnan(logits).any(), 'NaN in logits!'
assert not torch.isinf(logits).any(), 'Inf in logits!'

print('Forward pass sanity check:')
print(f'  Input  shape : {tuple(dummy_input.shape)}')
print(f'  Output shape : {tuple(logits.shape)}')
print(f'  Output dtype : {logits.dtype}')
print(f'  Logit range  : [{logits.min().item():.3f}, {logits.max().item():.3f}]')
print(f'  NaN/Inf      : None')
print('✓ Forward pass passed')

---
## 3. Parameter Count + FLOPs

In [ ]:
# ── Param count per layer group ───────────────────────────────────────────────
def count_params_by_group(model):
    groups = {'stem': 0, 'layer1': 0, 'layer2': 0,
              'layer3': 0, 'layer4': 0, 'head': 0}
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        for group in groups:
            if name.startswith(group):
                groups[group] += param.numel()
                break
    return groups

total  = model.count_params()
groups = count_params_by_group(model)

print(f'{'Layer':<12} {'Params':>12} {'%':>8}')
print('-' * 34)
for g, n in groups.items():
    print(f'{g:<12} {n:>12,} {100*n/total:>7.1f}%')
print('-' * 34)
print(f'{'TOTAL':<12} {total:>12,} 100.0%')
print(f'\nModel size (fp32): ~{total * 4 / 1e6:.1f} MB')

In [ ]:
# ── FLOPs estimation ──────────────────────────────────────────────────────────
# Install if needed: pip install thop
try:
    from thop import profile, clever_format
    flops, params = profile(model.cpu(), inputs=(torch.randn(1, 3, IMG_SIZE, IMG_SIZE),), verbose=False)
    flops_str, params_str = clever_format([flops, params], '%.2f')
    print(f'FLOPs  : {flops_str}')
    print(f'Params : {params_str}')
    model.to(device)
except ImportError:
    print('thop not installed — install with: pip install thop')
    print(f'Params : {total:,}')
    model.to(device)

---
## 4. Stage Output Shapes

In [ ]:
# ── Stage shapes ─────────────────────────────────────────────────────────────
shapes = model.get_stage_output_shapes(IMG_SIZE)

print(f'Stage output shapes (input: 1×3×{IMG_SIZE}×{IMG_SIZE}):')
print(f'  {'Stage':<10} {'Shape':>30}')
print('  ' + '-' * 42)
for stage, shape in shapes.items():
    print(f'  {stage:<10} {str(shape):>30}')

---
## 5. Architecture Diagram

In [ ]:
# ── Architecture diagram ──────────────────────────────────────────────────────
depth    = MCFG['arch']['depth']
channels = MCFG['arch']['channels']
se_on    = MCFG['blocks']['se_block']['enabled']

from models.resnet import RESNET_CONFIGS
_, stage_layers = RESNET_CONFIGS[depth]

fig, ax = plt.subplots(figsize=(16, 5))
ax.set_xlim(0, 16)
ax.set_ylim(0, 6)
ax.axis('off')

stage_info = [
    ('Input\n3×224×224',        0.4,  '#95a5a6'),
    (f'Stem\nConv7×7\n{channels[0]}×56×56', 2.0, '#3498db'),
    (f'Layer1\n{stage_layers[0]} blocks\n{channels[0]}×56×56', 4.2, '#2ecc71'),
    (f'Layer2\n{stage_layers[1]} blocks\n{channels[1]}×28×28', 6.4, '#2ecc71'),
    (f'Layer3\n{stage_layers[2]} blocks\n{channels[2]}×14×14', 8.6, '#2ecc71'),
    (f'Layer4\n{stage_layers[3]} blocks\n{channels[3]}×7×7',   10.8,'#2ecc71'),
    ('GAP\n512×1×1',            12.6, '#e67e22'),
    (f'FC\n{DCFG["dataset"]["num_classes"]} classes', 14.4, '#e74c3c'),
]

box_w, box_h = 1.5, 2.0
y_center = 3.0

for label, x, color in stage_info:
    rect = mpatches.FancyBboxPatch(
        (x - box_w/2, y_center - box_h/2), box_w, box_h,
        boxstyle='round,pad=0.1', linewidth=1.5,
        edgecolor='white', facecolor=color, alpha=0.88
    )
    ax.add_patch(rect)
    ax.text(x, y_center, label, ha='center', va='center',
            fontsize=8, fontweight='bold', color='white', wrap=True)

    # Arrow to next
    if x < 14.4:
        ax.annotate('', xy=(x + box_w/2 + 0.3, y_center),
                    xytext=(x + box_w/2, y_center),
                    arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.5))

# SE badge on layers 1-4
if se_on:
    for _, x, _ in stage_info[2:6]:
        ax.text(x, y_center + box_h/2 + 0.25, 'SE',
                ha='center', va='bottom', fontsize=8,
                color='#f39c12', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#fef9e7',
                          edgecolor='#f39c12', linewidth=1))

# Grad-CAM label
ax.text(10.8, y_center - box_h/2 - 0.35, '← Grad-CAM hook',
        ha='center', va='top', fontsize=8, color='#8e44ad', style='italic')

ax.set_title(
    f'ResNet-{depth} Architecture (SE={se_on}) | '
    f'Input: 224×224 | Classes: {DCFG["dataset"]["num_classes"]} | '
    f'Params: {total/1e6:.1f}M',
    fontsize=12, fontweight='bold', pad=15
)

plt.tight_layout()
Path('../docs').mkdir(exist_ok=True)
plt.savefig('../docs/architecture.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved → docs/architecture.png')

---
## 6. SE Block Visualisation

In [ ]:
# ── SE block weight distribution ─────────────────────────────────────────────
# Visualise what the SE block learns after a few random forward passes
from models.blocks import SEBlock

# Run 10 random forward passes, collect SE weights from first SE block
se_weights_all = []
model.eval()

# Hook into the first SE block in layer1
se_activations = []
def se_hook(module, input, output):
    se_activations.append(output.detach().cpu())

# Find first SE block
first_se = None
for module in model.modules():
    if isinstance(module, SEBlock):
        first_se = module
        break

if first_se and se_on:
    hook = first_se.register_forward_hook(se_hook)
    with torch.no_grad():
        for _ in range(10):
            x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
            _ = model(x)
    hook.remove()

    all_weights = torch.cat(se_activations, dim=0).squeeze()  # (10, C)
    mean_weights = all_weights.mean(0).numpy()  # (C,)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Channel weight distribution
    axes[0].bar(range(len(mean_weights)), mean_weights,
                color=plt.cm.RdYlGn(mean_weights), edgecolor='none')
    axes[0].set_xlabel('Channel Index')
    axes[0].set_ylabel('SE Weight (0–1)')
    axes[0].set_title('SE Block Channel Weights\n(mean over 10 random inputs)')
    axes[0].axhline(0.5, color='black', linestyle='--', linewidth=1, label='0.5 baseline')
    axes[0].legend()

    # Histogram of weights
    axes[1].hist(mean_weights, bins=20, color='#3498db', edgecolor='white')
    axes[1].set_xlabel('SE Weight Value')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('SE Weight Distribution\n(uniform early = not yet trained)')

    plt.suptitle('SE Block Analysis (Layer1, Block1)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../logs/se_block_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Note: Weights near 0.5 = uniform (untrained). After training, some channels')
    print('will be suppressed (<0.2) and others amplified (>0.8) — that is SE learning.')
else:
    print('SE blocks disabled in config — skipping SE visualisation')

---
## 7. Weight Initialisation Check

In [ ]:
# ── Weight init verification ──────────────────────────────────────────────────
conv_stds, bn_weights, bn_biases = [], [], []

for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        conv_stds.append(module.weight.data.std().item())
    elif isinstance(module, nn.BatchNorm2d):
        bn_weights.append(module.weight.data.mean().item())
        bn_biases.append(module.bias.data.mean().item())

print('Weight initialisation check:')
print(f'  Conv weight std   — mean: {np.mean(conv_stds):.4f}, std: {np.std(conv_stds):.4f}')
print(f'  (Kaiming He: std ≈ sqrt(2/fan_in) — varies by layer size)')
print()
print(f'  BN gamma mean     : {np.mean(bn_weights):.4f}  (expected ~1.0)')
print(f'  BN beta  mean     : {np.mean(bn_biases):.4f}  (expected ~0.0)')

# Zero-init check on last BN of residual blocks
from models.blocks import BasicBlock, Bottleneck
zero_init_count = 0
for module in model.modules():
    if isinstance(module, BasicBlock):
        if module.bn2.weight.data.abs().max().item() < 1e-6:
            zero_init_count += 1
    elif isinstance(module, Bottleneck):
        if module.bn3.weight.data.abs().max().item() < 1e-6:
            zero_init_count += 1

print(f'\n  Zero-init residual BN count: {zero_init_count}')
if zero_init_count > 0:
    print('  ✓ Zero-init applied — residual branches start as identity')
else:
    print('  ⚠ Zero-init not applied (check zero_init_residual in config)')

---
## 8. Grad-CAM Hook Verification

In [ ]:
# ── Grad-CAM hook test ────────────────────────────────────────────────────────
model.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)

with torch.no_grad():
    _ = model(dummy)

features = model.get_gradcam_features()
expected_spatial = IMG_SIZE // 32  # 224//32 = 7

assert features.shape[2] == expected_spatial, \
    f'Expected spatial {expected_spatial}, got {features.shape[2]}'

print('Grad-CAM hook verification:')
print(f'  layer4 feature shape : {tuple(features.shape)}')
print(f'  Expected             : (1, {MCFG["arch"]["channels"][-1]}, {expected_spatial}, {expected_spatial})')
print(f'  Hook registered      : {model._gradcam_hook is not None}')
print('✓ Grad-CAM hook working correctly')
print()
print('Note: After torch.load(), always call utils/checkpoint.load_checkpoint_with_hooks()')
print('to re-register this hook — otherwise heatmaps will be blank (v3 fix).')

---
## 9. Ablation Preview — ResNet-18 vs 34 vs 50 Params

In [ ]:
# ── Compare param counts across depths ───────────────────────────────────────
from models.resnet import ResNet

num_classes = DCFG['dataset']['num_classes']
channels    = MCFG['arch']['channels']
variants    = []

for depth in [18, 34, 50]:
    for se in [False, True]:
        m = ResNet(
            num_classes=num_classes,
            depth=depth,
            channels=channels,
            se_block=se,
        )
        variants.append({
            'name':   f'ResNet-{depth} SE={se}',
            'depth':  depth,
            'se':     se,
            'params': m.count_params(),
            'mb':     m.count_params() * 4 / 1e6,
        })

print(f'{'Model':<22} {'Params':>12} {'Size (MB)':>10}')
print('-' * 46)
for v in variants:
    marker = ' ← current' if (v['depth'] == MCFG['arch']['depth'] and
                               v['se'] == MCFG['blocks']['se_block']['enabled']) else ''
    print(f'{v["name"]:<22} {v["params"]:>12,} {v["mb"]:>9.1f}MB{marker}')

# Bar chart
fig, ax = plt.subplots(figsize=(11, 4))
names  = [v['name'] for v in variants]
params = [v['params']/1e6 for v in variants]
colors = ['#3498db' if not v['se'] else '#e74c3c' for v in variants]

bars = ax.bar(names, params, color=colors, edgecolor='white', linewidth=0.5)
ax.set_ylabel('Parameters (M)')
ax.set_title('ResNet Variant Comparison — Parameter Count')
ax.tick_params(axis='x', rotation=30)

for bar, val in zip(bars, params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}M', ha='center', va='bottom', fontsize=9)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='#3498db', label='Without SE'),
    Patch(facecolor='#e74c3c', label='With SE'),
], loc='upper left')

plt.tight_layout()
plt.savefig('../logs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Day 2 Summary

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
from models.resnet import RESNET_CONFIGS
_, slayers = RESNET_CONFIGS[MCFG['arch']['depth']]

print('=' * 60)
print('DAY 2 SUMMARY')
print('=' * 60)
print(f'  Model         : ResNet-{MCFG["arch"]["depth"]} + SE blocks')
print(f'  Stages        : {slayers}')
print(f'  Channels      : {MCFG["arch"]["channels"]}')
print(f'  SE enabled    : {MCFG["blocks"]["se_block"]["enabled"]}')
print(f'  Params        : {model.count_params():,}')
print(f'  Model size    : ~{model.count_params()*4/1e6:.1f} MB (fp32)')
print(f'  num_classes   : {DCFG["dataset"]["num_classes"]} (from data_config.yaml)')
print()
print('Plots saved to:')
print('  docs/architecture.png')
print('  logs/se_block_analysis.png')
print('  logs/model_comparison.png')
print()
print('Interview answer:')
print('  "I built ResNet from scratch with SE blocks for channel attention.')
print('   num_classes reads from data_config.yaml — switching datasets')
print('   requires zero code changes. Zero-init on residual BNs improves')
print('   training stability per He et al. Bag of Tricks."')